# Run a LangGraph BYO agent

This notebook deploys `langgraph-researcher`, a custom LangGraph workflow packaged as a kagent BYO agent, from a prebuilt image. See the [README](./README.md) for how the workflow works and how to build your own image.

kagent must be enabled on your prokube cluster, and you need an Anthropic API key.

## Setup

[api_key.py](../../src/pk_helpers/api_key.py) and [kagent.py](../../src/pk_helpers/kagent.py) are repository helpers, not built-in Python functions. `%run -n` brings their functions into the notebook namespace without running CLI entry points guarded by `if __name__ == "__main__"`.

In [ ]:
# Supplies get_or_create_anthropic_api_key and get_or_create_a2a_api_key.
%run -n ../../src/pk_helpers/api_key.py
# Supplies workspace_a2a_url, agent_a2a_url, and send_message.
%run -n ../../src/pk_helpers/kagent.py

In [ ]:
import subprocess
from pathlib import Path

with open("/var/run/secrets/kubernetes.io/serviceaccount/namespace") as f:
    namespace = f.read().strip()


def kubectl_apply(manifest: str) -> None:
    subprocess.run(["kubectl", "apply", "-f", "-"], input=manifest, text=True, check=True)

## Create the credentials

The agents call Claude with the key in the `anthropic-api-key` Secret. The next cell creates the Secret unless it already exists, for example because you created it on the **Secrets** page of the prokube UI. It prompts for the key unless `ANTHROPIC_API_KEY` is set.

In [ ]:
secret_exists = subprocess.run(
    ["kubectl", "get", "secret", "anthropic-api-key"], capture_output=True
).returncode == 0
if not secret_exists:
    kubectl_apply(
        Path("../basic-agent/anthropic-secret.yaml")
        .read_text()
        .replace("<your-anthropic-api-key>", get_or_create_anthropic_api_key())
    )

## Deploy

In [ ]:
!kubectl apply -f agent.yaml
!kubectl wait --for=condition=Ready agents.kagent.dev/langgraph-researcher --timeout=3m

## Talk to the agent from this Lab

Labs reach agents through the internal Agent Gateway route without an API key. You can also open **Agents** in the prokube UI, select `langgraph-researcher`, and chat there.

In [ ]:
internal_url = workspace_a2a_url(namespace, "langgraph-researcher")
print(internal_url)

In [ ]:
prompt = "Research https://prokube.ai and summarize what prokube offers in one sentence. Include the source URL."
print(send_message(internal_url, prompt))

## Connect with an API key

Clients outside the cluster use the agent's external route, `https://<your-prokube-domain>/svc/a2a/<workspace>/langgraph-researcher`, with an API key. Create a key with access to `langgraph-researcher` on the **API Keys** page at `https://<your-prokube-domain>/pkui/ai-gateway/keys`.

This uses the same `message/send` operation and prompt, but a different route with API-key authentication. This notebook calls `/svc/a2a` on the in-cluster host; external clients use the HTTPS pattern above. The next cells show the URL, then prompt for the key unless `A2A_API_KEY` is set and send the message.

In [ ]:
api_key_url = agent_a2a_url(namespace, "langgraph-researcher")
print(api_key_url)

In [ ]:
api_key = get_or_create_a2a_api_key()
print(send_message(api_key_url, prompt, api_key=api_key))

## Clean up

Also delete the Secret with `kubectl delete secret anthropic-api-key` if no other agent in the workspace uses it.

In [ ]:
!kubectl delete -f agent.yaml